# App Inventory Signal Analysis

재고 검색/재고 상세 조회가 POS 판매 전 선행 관심 신호로 쓸 수 있는지 확인하는 노트북입니다.

- 입력: `data/processed/app_event_integrated/*.parquet`, `data/raw/app/상품코드목록_260416.csv`, `data/processed/B2_POS_SALE.parquet`
- 출력: `eda/app_event_yumi/outputs/inventory_signal_analysis/`
- 주의: 재고 이벤트에는 직접 상품 ID가 없어서, 상품 단위 분석은 `재고 검색어 -> 상품명 후보 매칭`으로만 가능합니다.

## 0. 환경 준비

분석 모듈을 불러오고 출력 폴더를 준비합니다.

In [ ]:
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import Image, display

BASE_DIR = Path.cwd()
if not (BASE_DIR / "app_inventory_signal_analysis.py").exists():
    BASE_DIR = Path.cwd() / "eda" / "app_event_yumi"

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

analysis = importlib.import_module("app_inventory_signal_analysis")
analysis = importlib.reload(analysis)

OUTPUT_DIR = analysis.OUTPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", font="AppleGothic")
plt.rcParams["axes.unicode_minus"] = False

BASE_DIR, OUTPUT_DIR

## 1. 재고 이벤트 로드

`af_search_inventory`, `af_content_view_inventory`만 필터링하고 JSON 안의 `af_search_string`을 꺼냅니다.

In [ ]:
lf = analysis.app_event_lf()
inv = analysis.inventory_lf(lf)
inv.collect_schema()

## 2. 재고 이벤트 로깅 품질

상품 ID가 있는지, 검색어가 있는지, placeholder 사용자 비중이 어느 정도인지 확인합니다.

In [ ]:
summary = analysis.collect_inventory_event_summary(inv)
analysis.save_csv(summary, "inventory_event_summary.csv")
summary

## 3. 일자별 재고 퍼널

일자별 재고 검색과 재고 상세 조회 규모를 보고, `상세 조회 / 검색` 비율을 확인합니다.

In [ ]:
daily_funnel = analysis.collect_daily_inventory_funnel(inv)
analysis.save_csv(daily_funnel, "daily_inventory_funnel.csv")
daily_funnel.head(20)

In [ ]:
daily_funnel_pd = daily_funnel.to_pandas()
ax = daily_funnel_pd.plot(
    x="event_date",
    y=["event_count_af_search_inventory", "event_count_af_content_view_inventory"],
    figsize=(14, 5),
)
ax.set_title("Daily Inventory Search and Inventory Detail Views")
ax.set_xlabel("Date")
ax.set_ylabel("Event count")
plt.tight_layout()
plt.show()

## 4. 재고 검색어 랭킹

비정상 사용자 `L00000000000`을 제외하고 재고 검색어를 집계합니다. 이 검색어가 상품 후보 매칭의 출발점입니다.

In [ ]:
search_terms = analysis.collect_search_terms(inv)
analysis.save_csv(search_terms, "inventory_search_terms.csv")
analysis.save_csv(search_terms.head(200), "top200_inventory_search_terms.csv")
search_terms.head(50)

In [ ]:
analysis.save_barplot(
    search_terms.head(30),
    x="search_count",
    y="inventory_search_string",
    title="Top 30 Inventory Search Terms",
    xlabel="Search count",
    ylabel="Search term",
    filename="top30_inventory_search_terms.png",
)
display(Image(filename=str(OUTPUT_DIR / "top30_inventory_search_terms.png")))

## 5. 검색어-상품명 후보 매칭

상위 검색어를 상품명과 보수적으로 매칭합니다. `match_type`, `match_score`를 반드시 같이 확인해야 합니다.

In [ ]:
mapping = analysis.load_mapping()
candidates = analysis.match_search_terms_to_products(search_terms, mapping)
best_matches = analysis.best_product_matches(candidates)

analysis.save_csv(candidates, "inventory_search_product_candidates.csv")
analysis.save_csv(best_matches, "inventory_search_best_product_matches.csv")

best_matches.head(50)

## 6. 매칭 커버리지

상위 검색어 중 상품 후보로 연결된 비중을 확인합니다. 커버리지가 낮으면 재고 검색은 상품 단위보다 키워드/콘텐츠 단위로 쓰는 편이 안전합니다.

In [ ]:
top_terms = search_terms.head(analysis.DEFAULT_TOP_TERMS)
coverage = top_terms.select(
    [
        analysis.pl.len().alias("top_search_terms"),
        analysis.pl.col("search_count").sum().alias("top_term_searches"),
    ]
).with_columns(
    [
        analysis.pl.lit(best_matches.height).alias("matched_terms"),
        analysis.pl.lit(best_matches["search_count"].sum() if not best_matches.is_empty() else 0).alias(
            "matched_term_searches"
        ),
    ]
).with_columns(
    [
        (analysis.pl.col("matched_terms") / analysis.pl.col("top_search_terms") * 100).round(4).alias(
            "matched_term_pct"
        ),
        (analysis.pl.col("matched_term_searches") / analysis.pl.col("top_term_searches") * 100).round(4).alias(
            "matched_search_pct"
        ),
    ]
)
coverage

## 7. 상품 후보별 일자 재고 검색량

매칭된 검색어를 상품 후보 단위로 일자별 집계합니다.

In [ ]:
daily_search_terms = analysis.collect_daily_search_terms(inv, search_terms.head(analysis.DEFAULT_TOP_TERMS))
matched_daily_searches = analysis.collect_matched_daily_searches(daily_search_terms, best_matches)

analysis.save_csv(daily_search_terms, "daily_top_inventory_search_terms.csv")
analysis.save_csv(matched_daily_searches, "daily_inventory_searches_matched_products.csv")

matched_daily_searches.head(50)

## 8. POS 판매와 연결

매칭된 상품 후보만 POS에서 일자별 판매량을 집계합니다. 원천 POS parquet은 읽기만 합니다.

In [ ]:
if matched_daily_searches.is_empty():
    pos_daily_sales = analysis.pl.DataFrame()
else:
    start_date = matched_daily_searches["event_date"].min()
    end_date = matched_daily_searches["event_date"].max()
    product_codes = matched_daily_searches["mapped_pos_item_code"].unique().to_list()
    pos_daily_sales = analysis.collect_pos_daily_sales(product_codes, start_date, end_date)

analysis.save_csv(pos_daily_sales, "pos_daily_sales_for_inventory_matched_products.csv")
pos_daily_sales.head(50)

## 9. Lead-Lag 상관 확인

`lag_days=1`은 오늘 재고 검색량과 내일 POS 판매량의 상관입니다. 상품명 후보 매칭 기반이므로, 높은 상관은 추가 검증 후보로 해석합니다.

In [ ]:
lead_lag = analysis.collect_lead_lag_correlations(matched_daily_searches, pos_daily_sales)
analysis.save_csv(lead_lag, "inventory_search_pos_lead_lag_correlations.csv")
if not lead_lag.is_empty():
    analysis.save_csv(
        lead_lag.filter(analysis.pl.col("lag_days") > 0).head(100),
        "top100_positive_lag_inventory_search_pos_correlations.csv",
    )
lead_lag.head(50)

## 10. 선행 신호 후보만 보기

당일 상관보다 `lag_days > 0`에서 높게 나오는 상품을 우선 검토합니다. `paired_days`, `total_searches`, `total_sales_qty`가 너무 작으면 제외하는 것이 좋습니다.

In [ ]:
lead_candidates = (
    lead_lag.filter(
        (analysis.pl.col("lag_days") > 0)
        & (analysis.pl.col("paired_days") >= 14)
        & (analysis.pl.col("total_searches") >= 100)
        & (analysis.pl.col("total_sales_qty") >= 50)
    )
    .sort(["search_sales_qty_corr", "total_searches"], descending=[True, True])
)
lead_candidates.head(50)

## 11. 특정 상품 시계열 확인

위 후보 중 하나를 골라 재고 검색량과 POS 판매량을 같은 날짜 축에서 확인합니다.

In [ ]:
if lead_candidates.is_empty():
    print("No lead candidates after filters.")
else:
    selected_code = lead_candidates.item(0, "mapped_pos_item_code")
    selected_name = lead_candidates.item(0, "mapped_item_name")
    ts = (
        matched_daily_searches.filter(analysis.pl.col("mapped_pos_item_code") == selected_code)
        .select(["event_date", "inventory_search_count"])
        .join(
            pos_daily_sales.filter(analysis.pl.col("mapped_pos_item_code") == selected_code).select(
                ["event_date", "pos_sales_qty"]
            ),
            on="event_date",
            how="outer",
            coalesce=True,
        )
        .fill_null(0)
        .sort("event_date")
        .to_pandas()
    )
    ax = ts.plot(x="event_date", y=["inventory_search_count", "pos_sales_qty"], figsize=(14, 5))
    ax.set_title(f"Inventory Search vs POS Sales: {selected_name} ({selected_code})")
    ax.set_xlabel("Date")
    ax.set_ylabel("Count / Qty")
    plt.tight_layout()
    plt.show()

## 12. 해석 메모

현재 로그 기준으로는 재고 상세 조회 이벤트에 상품 ID가 없습니다. 따라서 가장 확실한 다음 개선점은 앱 로그에 `af_content_id` 또는 POS 상품코드, 점포코드를 재고 상세 조회 이벤트에도 남기는 것입니다. 그 전까지는 재고 검색어 기반 후보 매칭으로 선행 신호 가능성을 탐색하는 수준이 안전합니다.